# Check off-pair probabilities against the PDF

For each selected dataset case, this notebook compares `actual_node_probabilities(...).keep_i` with the independent closed-form value of $Q^k_{ijmn}$ from equations (2)–(7) of `Q_offpair_probability_proof.pdf`. Here $k\notin\{i,j\}$, and outcome $i$ means the branch that discards $j$.

The first three cases from each compact benchmark are checked at every location and at every sensor count stored in their CSV files. The notebook prints the maximum difference and one example from each regime. A tolerance of `1e-10` is used for floating-point agreement.


In [1]:
from collections import Counter
from math import ceil, cos, isclose, pi, sin
from pathlib import Path
import sys

# Works when Jupyter starts from the repository root or this folder.
project = next((p for p in (Path.cwd(), Path.cwd() / "min-error-sequential")
                if (p / "src" / "actual_error_analysis.py").exists()), None)
if project is None:
    raise FileNotFoundError("Start Jupyter from the repository root or min-error-sequential folder")
sys.path.insert(0, str(project / "src"))

from actual_error_analysis import actual_node_probabilities
from input_data import load_sparse_input, principal_phase

DATASETS = ("compact_benchmark_1d", "compact_benchmark_2d")
CASES_PER_DATASET = 3
TOLERANCE = 1e-10


## PDF equations

The function below implements the PDF's five cases directly from the three pairwise phase gaps. It does not construct states or diagonalize a Helstrom operator. The existing project function does construct and diagonalize that operator.


In [2]:
def signed_gap(a, b, m, phase_gaps):
    if a == b:
        return 0.0
    if a < b:
        return phase_gaps[(a, b)][m]
    return principal_phase(-phase_gaps[(b, a)][m])


def q_from_pdf(i, j, k, m, n, phase_gaps):
    rij = signed_gap(i, j, m, phase_gaps)
    rik = signed_gap(i, k, m, phase_gaps)
    rjk = signed_gap(j, k, m, phase_gaps)
    gap = abs(rij)

    if n == 0 or gap == 0:
        return 0.5, "no information (2)"

    s = ceil(pi / gap)
    if n < s:
        # PDF equation (3).
        q = 0.5 + (cos(n * rik / 2)**2 - cos(n * rjk / 2)**2) / (2 * sin(n * gap / 2))
        return q, "below threshold (3)"

    if isclose(s * gap, pi, rel_tol=0, abs_tol=1e-12):
        # PDF equation (5); only s sensors are used even when n > s.
        ai = cos(s * rik / 2)
        aj = cos(s * rjk / 2)
        regime = "exact threshold (5)"
    elif s % 2 == 0:
        # PDF equation (6).
        c = cos(s * gap / 2)
        ai = (cos(s * rik / 2) - c) / (1 - c)
        aj = (cos(s * rjk / 2) - c) / (1 - c)
        regime = "perfect, even s (6)"
    else:
        # PDF equation (7).
        c1 = cos(gap / 2)
        cs = cos(s * gap / 2)
        ai = (c1 * cos(s * rik / 2) - cs * cos(rik / 2)) / (c1 - cs)
        aj = (c1 * cos(s * rjk / 2) - cs * cos(rjk / 2)) / (c1 - cs)
        regime = "perfect, odd s (7)"

    return 0.5 + 0.5 * (ai**2 - aj**2), regime


In [3]:
rows = []
for dataset in DATASETS:
    paths = sorted((project / "inputs" / dataset).glob("case_*.csv"))[:CASES_PER_DATASET]
    if not paths:
        raise FileNotFoundError(f"No input CSV files found for {dataset}")
    for path in paths:
        K, M, phase_gaps, pair_errors = load_sparse_input(path)
        for i in range(K):
            for j in range(i + 1, K):
                for k in range(K):
                    if k in (i, j):
                        continue
                    for m in range(M):
                        for n in sorted(pair_errors[(i, j)][m]):
                            pdf_q, regime = q_from_pdf(i, j, k, m, n, phase_gaps)
                            code_q = actual_node_probabilities(
                                (i, j), m, n, k, phase_gaps
                            ).keep_i
                            rows.append({
                                "dataset": dataset, "case": path.stem,
                                "i": i, "j": j, "k": k, "m": m, "n": n,
                                "regime": regime, "code_q": code_q,
                                "pdf_q": pdf_q, "difference": abs(code_q - pdf_q),
                            })

counts = Counter(row["regime"] for row in rows)
max_difference = max(row["difference"] for row in rows)
print(f"Compared {len(rows)} off-pair calculations from {len(DATASETS) * CASES_PER_DATASET} dataset cases")
print("Regime counts:", dict(counts))
print(f"Maximum absolute difference: {max_difference:.3e}")
print("AGREE" if max_difference <= TOLERANCE else "DISAGREE")

print("\nOne example per regime (code Q, PDF Q, absolute difference):")
for regime in counts:
    row = next(row for row in rows if row["regime"] == regime)
    print(f"{row['dataset']} {row['case']} (i,j,k,m,n)="
          f"({row['i']},{row['j']},{row['k']},{row['m']},{row['n']}) "
          f"{regime}: {row['code_q']:.12f}, {row['pdf_q']:.12f}, "
          f"{row['difference']:.3e}")

if max_difference > TOLERANCE:
    print("\nLargest disagreements:")
    for row in sorted(rows, key=lambda row: row["difference"], reverse=True)[:10]:
        print(row)
assert max_difference <= TOLERANCE, "The code and PDF formulas disagree"


Compared 549 off-pair calculations from 6 dataset cases
Regime counts: {'no information (2)': 54, 'below threshold (3)': 373, 'perfect, even s (6)': 72, 'perfect, odd s (7)': 50}
Maximum absolute difference: 8.105e-15
AGREE

One example per regime (code Q, PDF Q, absolute difference):
compact_benchmark_1d case_00001 (i,j,k,m,n)=(0,1,2,0,0) no information (2): 0.500000000000, 0.500000000000, 0.000e+00
compact_benchmark_1d case_00001 (i,j,k,m,n)=(0,1,2,0,1) below threshold (3): 0.537825572800, 0.537825572800, 2.220e-16
compact_benchmark_1d case_00001 (i,j,k,m,n)=(0,1,2,0,4) perfect, even s (6): 0.618858454275, 0.618858454275, 4.441e-16
compact_benchmark_1d case_00001 (i,j,k,m,n)=(0,1,2,2,5) perfect, odd s (7): 0.968785804176, 0.968785804176, 8.882e-16


The generated compact data generally have nonzero, non-exact pair gaps, so equations (2), (3), (6), and (7) appear in the dataset check. The next cell checks the two exact-threshold edge cases with consistent three-field phases.


In [4]:
edge_cases = {
    "identical tested fields": {(0, 1): [0.0], (0, 2): [0.37], (1, 2): [0.37]},
    "exact threshold": {(0, 1): [pi / 2], (0, 2): [0.37], (1, 2): [0.37 - pi / 2]},
}
for label, phases in edge_cases.items():
    pdf_q, regime = q_from_pdf(0, 1, 2, 0, 2, phases)
    code_q = actual_node_probabilities((0, 1), 0, 2, 2, phases).keep_i
    difference = abs(code_q - pdf_q)
    print(f"{label}: {regime}; code={code_q:.12f}, PDF={pdf_q:.12f}, difference={difference:.3e}")
    assert difference <= TOLERANCE


identical tested fields: no information (2); code=0.500000000000, PDF=0.500000000000, difference=0.000e+00
exact threshold: exact threshold (5); code=0.869234279365, PDF=0.869234279365, difference=1.110e-16
